# 02 – Datenbereinigung

> Hinweis: Dieses Notebook ist ein exploratives Arbeitsartefakt. Der aktuelle, benotungsrelevante Stand steht in `../reports/FINAL_REPORT.pdf`; alte Zelloutputs wurden entfernt und die Zellen sollen bei Bedarf neu ausgefuehrt werden.


**Projekt:** Polymarket Reddit Sentiment  
**Kurs:** Data Wrangling & Engineering (FHNW)

Dieses Notebook behandelt:
1. Fehlende Werte (Typ-Analyse: MCAR / MAR / MNAR)
2. Duplikate entfernen
3. Ausreisser erkennen und behandeln (z-Score, IQR, Winsorisierung)
4. Textreinigung
5. Qualitätsprüfung
6. Bereinigten Datensatz speichern

---

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print('Libraries geladen.')

In [ ]:
# Rohdaten laden (aus Notebook 01 gespeichert)
import os

if os.path.exists('../data/reddit_raw.csv'):
    posts_df = pd.read_csv('../data/reddit_raw.csv', parse_dates=['created_utc'])
    markets_df = pd.read_csv('../data/polymarket_raw.csv')
    print(f'Reddit:     {posts_df.shape}')
    print(f'Polymarket: {markets_df.shape}')
else:
    # Fallback: direkt laden
    from src import reddit, polymarket, sentiment
    raw = reddit.get_posts('Bitcoin', ['investing', 'stocks', 'worldnews'], 100)
    posts_df = sentiment.analyze(raw)
    try:
        markets_df = polymarket.get_markets(50)
    except:
        markets_df = pd.DataFrame({'id':[], 'question':[], 'probability':[], 'category':[], 'volume':[]})
    print('Daten direkt geladen (kein CSV vorhanden).')

## 1. Fehlende Werte – Analyse

In [ ]:
def missing_report(df, name):
    total = len(df)
    report = pd.DataFrame({
        'Spalte': df.columns,
        'Fehlend (n)': df.isnull().sum().values,
        'Fehlend (%)': (df.isnull().mean() * 100).round(2).values,
        'Dtype': df.dtypes.values
    }).sort_values('Fehlend (n)', ascending=False)
    print(f'\n=== {name} (n={total}) ===')
    print(report.to_string(index=False))
    return report

report_posts   = missing_report(posts_df, 'Reddit Posts')
report_markets = missing_report(markets_df, 'Polymarket Märkte')

### 1.1 Typ der fehlenden Werte bestimmen

| Spalte | Vermuteter Typ | Begründung |
|---|---|---|
| `text` (leer) | **MCAR** | Link-Posts haben keinen Text-Body – zufällig, nicht systematisch |
| `probability` (Polymarket) | **MAR** | Fehlt nur bei geschlossenen oder archivierten Märkten |
| `category` (Polymarket) | **MAR** | Kategorisierung erfolgt manuell, fehlt bei neuen Märkten |

> **Konsequenz:** `text`-Spalte kann mit leerem String aufgefüllt werden (MCAR, geringe Verzerrung).  
> `probability` wird mit Median imputiert + Indikator-Variable erstellt (MAR-Strategie).

In [ ]:
posts_clean = posts_df.copy()
markets_clean = markets_df.copy()

# ── Reddit: text = MCAR → leerer String ────────────────────────────────────
n_text_missing = posts_clean['text'].isnull().sum()
posts_clean['text'] = posts_clean['text'].fillna('')
print(f'Reddit text: {n_text_missing} fehlende Werte mit "" aufgefüllt')

# ── Polymarket: probability = MAR → Median + Indikator ─────────────────────
if 'probability' in markets_clean.columns:
    n_prob_missing = markets_clean['probability'].isnull().sum()
    markets_clean['probability_missing'] = markets_clean['probability'].isnull().astype(int)
    median_prob = markets_clean['probability'].median()
    markets_clean['probability'] = markets_clean['probability'].fillna(median_prob)
    print(f'Polymarket probability: {n_prob_missing} Werte mit Median ({median_prob:.3f}) imputiert')
    print(f'Indikatorvariable "probability_missing" erstellt')

# ── Polymarket: category → 'Unknown' ───────────────────────────────────────
if 'category' in markets_clean.columns:
    n_cat_missing = markets_clean['category'].isnull().sum()
    markets_clean['category'] = markets_clean['category'].fillna('Unknown')
    print(f'Polymarket category: {n_cat_missing} Werte mit "Unknown" aufgefüllt')

In [ ]:
# Validation: Keine fehlenden Werte mehr in kritischen Spalten
critical_cols_posts = ['id', 'title', 'subreddit', 'compound', 'sentiment_label']
existing = [c for c in critical_cols_posts if c in posts_clean.columns]
remaining = posts_clean[existing].isnull().sum()
print('Verbleibende fehlende Werte in kritischen Spalten (Reddit):')
print(remaining)
assert remaining.sum() == 0, 'Es gibt noch fehlende Werte in kritischen Spalten!'

## 2. Duplikate entfernen

In [ ]:
before = len(posts_clean)
posts_clean = posts_clean.drop_duplicates(subset='id', keep='first')
after_id = len(posts_clean)
print(f'Duplikate (nach id) entfernt: {before - after_id}')

# Zusätzlich: Posts mit identischem Titel (Crossposts)
before2 = len(posts_clean)
posts_clean = posts_clean.drop_duplicates(subset='title', keep='first')
after_title = len(posts_clean)
print(f'Duplikate (nach title) entfernt: {before2 - after_title}')
print(f'Endgrösse: {len(posts_clean)} Posts')

## 3. Ausreisser erkennen

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].boxplot(posts_clean['score'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[0].set_title('Boxplot: Reddit Score')
axes[0].set_ylabel('Score')

axes[1].boxplot(posts_clean['num_comments'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='darkorange', alpha=0.6))
axes[1].set_title('Boxplot: Anzahl Kommentare')
axes[1].set_ylabel('Anzahl Kommentare')

plt.tight_layout()
plt.show()

In [ ]:
# ── Methode 1: IQR-Methode ──────────────────────────────────────────────────
def iqr_outliers(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    mask = (series < lower) | (series > upper)
    return mask, lower, upper

# ── Methode 2: z-Score ──────────────────────────────────────────────────────
def zscore_outliers(series, threshold=3):
    z = np.abs(stats.zscore(series.dropna()))
    mask = pd.Series(False, index=series.index)
    mask.loc[series.dropna().index] = z > threshold
    return mask

for col in ['score', 'num_comments']:
    iqr_mask, lower, upper = iqr_outliers(posts_clean[col])
    z_mask = zscore_outliers(posts_clean[col])
    print(f'{col}:')
    print(f'  IQR-Methode:  {iqr_mask.sum()} Ausreisser  (< {lower:.0f} oder > {upper:.0f})')
    print(f'  z-Score (>3): {z_mask.sum()} Ausreisser')
    print()

### 3.1 Ausreisser-Strategie

| Spalte | Anteil Ausreisser | Strategie | Begründung |
|---|---|---|---|
| `score` | ~5-15% | **Winsorisierung** (95. Perzentile) | Viral-Posts sind echte Daten, nicht entfernen |
| `num_comments` | ~5-10% | **Winsorisierung** (95. Perzentile) | Gleiche Begründung |
| `compound` | — | **Behalten** | [-1, 1] ist definierter Wertebereich, keine Ausreisser möglich |

In [ ]:
# ── Winsorisierung ──────────────────────────────────────────────────────────
def winsorize(series, lower_pct=0.05, upper_pct=0.95):
    lower = series.quantile(lower_pct)
    upper = series.quantile(upper_pct)
    return series.clip(lower=lower, upper=upper), lower, upper

posts_clean['score_winsorized'], s_lo, s_hi = winsorize(posts_clean['score'])
posts_clean['num_comments_winsorized'], c_lo, c_hi = winsorize(posts_clean['num_comments'])

print(f'score: Winsorisiert auf [{s_lo:.0f}, {s_hi:.0f}]')
print(f'num_comments: Winsorisiert auf [{c_lo:.0f}, {c_hi:.0f}]')

# Vergleich Original vs. Winsorisiert
compare = pd.DataFrame({
    'score (original)':      [posts_clean['score'].min(), posts_clean['score'].max(), posts_clean['score'].mean()],
    'score (winsorisiert)':  [posts_clean['score_winsorized'].min(), posts_clean['score_winsorized'].max(), posts_clean['score_winsorized'].mean()],
}, index=['Min', 'Max', 'Mean'])
compare.round(2)

## 4. Textreinigung

In [ ]:
import re

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = re.sub(r'http\S+', '', text)          # URLs entfernen
    text = re.sub(r'\[deleted\]|\[removed\]', '', text)  # Reddit-Platzhalter
    text = re.sub(r'\s+', ' ', text).strip()     # Mehrfache Leerzeichen
    return text

posts_clean['title_clean'] = posts_clean['title'].apply(clean_text)
posts_clean['text_clean']  = posts_clean['text'].apply(clean_text)

# Posts ohne verwertbaren Inhalt markieren
posts_clean['is_empty_content'] = (
    (posts_clean['title_clean'].str.len() < 5) &
    (posts_clean['text_clean'].str.len() < 5)
).astype(int)

print(f'Posts ohne verwertbaren Inhalt: {posts_clean["is_empty_content"].sum()}')
print(f'Beispiele bereinigter Titel:')
posts_clean[['title', 'title_clean']].head(3)

## 5. Qualitätsprüfung

In [ ]:
print('=== Qualitätsprüfung: Reddit Posts ===')
checks = {
    'Keine fehlenden IDs':         posts_clean['id'].isnull().sum() == 0,
    'Keine fehlenden Titel':        posts_clean['title'].isnull().sum() == 0,
    'Compound im Wertebereich':    posts_clean['compound'].between(-1, 1).all(),
    'Sentiment-Labels vollständig': posts_clean['sentiment_label'].isin(['positive','neutral','negative']).all(),
    'Keine doppelten IDs':          posts_clean['id'].duplicated().sum() == 0,
    'Score >= 0 (nach Clip)':       (posts_clean['score'] >= 0).all(),
}

for check, passed in checks.items():
    icon = '✅' if passed else '❌'
    print(f'  {icon}  {check}')

all_passed = all(checks.values())
print(f'\nGesamt: {"ALLE CHECKS BESTANDEN" if all_passed else "FEHLER GEFUNDEN"}')

## 6. Bereinigten Datensatz speichern

In [ ]:
import os
os.makedirs('../data', exist_ok=True)

posts_clean.to_csv('../data/reddit_clean.csv', index=False)
markets_clean.to_csv('../data/polymarket_clean.csv', index=False)

print('Gespeichert:')
print(f'  data/reddit_clean.csv    ({len(posts_clean)} Zeilen, {posts_clean.shape[1]} Spalten)')
print(f'  data/polymarket_clean.csv ({len(markets_clean)} Zeilen, {markets_clean.shape[1]} Spalten)')

print('\nNeue Spalten nach Bereinigung:')
new_cols = [c for c in posts_clean.columns if c not in posts_df.columns]
print('  ' + ', '.join(new_cols))

## 7. Zusammenfassung Datenbereinigung

| Schritt | Massnahme | Begründung |
|---|---|---|
| **Fehlende `text`** | Leerstring (MCAR) | Zufällig, keine Verzerrung |
| **Fehlende `probability`** | Median-Imputation + Indikator | MAR, Indikator als Feature nutzbar |
| **Duplikate (id)** | Erstes Vorkommen behalten | Eindeutigkeit sicherstellen |
| **Duplikate (title)** | Erstes Vorkommen behalten | Crossposts vermeiden |
| **Ausreisser `score`** | Winsorisierung 5–95% | Virale Posts bleiben, Extremwerte gedämpft |
| **Ausreisser `num_comments`** | Winsorisierung 5–95% | Gleiche Strategie |
| **Textreinigung** | URLs/Platzhalter entfernt | Sentiment-Analyse verbessern |